# Use your annotated dataset

In [1]:
!pip install -q argilla datasets huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 6.2 MB/s eta 0:00:00


In [2]:
import os
import logging
import argilla as rg

logging.basicConfig(level=logging.INFO)

print("=" * 80)
print("Argilla imported successfully.")
print("Argilla version:", rg.__version__)
print("=" * 80)

/usr/local/lib/python3.12/dist-packages/argilla/_api/_token.py:83: UserWarning: 
The secrets ARGILLA_API_URL and does not exist in your Colab secrets.
  warnings.warn(f"\nThe secrets {name} and does not exist in your Colab secrets.")
/usr/local/lib/python3.12/dist-packages/argilla/_api/_token.py:83: UserWarning: 
The secrets ARGILLA_API_KEY and does not exist in your Colab secrets.
  warnings.warn(f"\nThe secrets {name} and does not exist in your Colab secrets.")


Argilla imported successfully.
Argilla version: 2.8.0


In [4]:
!pip install -q argilla huggingface_hub

In [7]:
import argilla as rg

HF_TOKEN = ""

# You create this yourself. It becomes the API key for your new Argilla server.
ARGILLA_API_KEY = "owner.apikey"

client = rg.Argilla.deploy_on_spaces(
    api_key=ARGILLA_API_KEY,
    hf_token=HF_TOKEN,
    repo_name="argilla-parth-test",
    private=False,
)

print("Argilla Space deployed.")
print(client)

Argilla Space deployed.


Argilla has been deployed at: https://parthgeek-argilla-parth-test.hf.space/


In [8]:
ARGILLA_API_URL = "https://parthgeek-argilla-parth-test.hf.space/"
ARGILLA_API_KEY = "owner.apikey"
HF_TOKEN = ""

print("=" * 80)
print("Credentials collected.")
print("Argilla URL:", ARGILLA_API_URL)
print("HF token provided:", bool(HF_TOKEN.strip()))
print("=" * 80)

Credentials collected.
Argilla URL: https://parthgeek-argilla-parth-test.hf.space/
HF token provided: False


In [10]:
import argilla as rg

ARGILLA_API_URL = "https://parthgeek-argilla-parth-test.hf.space/"
ARGILLA_API_KEY = "owner.apikey"

print("=" * 80)
print("Connecting to Argilla...")
print("=" * 80)

client = rg.Argilla(
    api_url=ARGILLA_API_URL,
    api_key=ARGILLA_API_KEY,
    headers={}
)

print("Connected to Argilla successfully.")
print(client)

Connecting to Argilla...
Connected to Argilla successfully.


Argilla has been deployed at: https://parthgeek-argilla-parth-test.hf.space/


In [12]:
from datasets import load_dataset
import argilla as rg

DATASET_NAME = "ag_news"

print("=" * 80)
print("Loading or creating Argilla dataset...")
print("=" * 80)

dataset = client.datasets(name=DATASET_NAME)

if dataset is None:
    print("Dataset does not exist. Creating new dataset...")

    settings = rg.Settings(
        guidelines="Classify the news article into one of four categories.",
        fields=[
            rg.TextField(
                name="text",
                title="News text"
            )
        ],
        questions=[
            rg.LabelQuestion(
                name="label",
                title="Which category does this news article belong to?",
                labels=["World", "Sports", "Business", "Sci/Tech"]
            )
        ]
    )

    dataset = rg.Dataset(
        name=DATASET_NAME,
        settings=settings,
        client=client
    )

    dataset.create()

    print("Dataset created successfully:", dataset.name)

else:
    print("Dataset already exists:", dataset.name)

print("=" * 80)
print("Loading AG News sample from Hugging Face Datasets...")
print("=" * 80)

ag_news = load_dataset("ag_news", split="train[:50]")

records = [
    rg.Record(
        fields={
            "text": row["text"]
        }
    )
    for row in ag_news
]

print("Logging records to Argilla without metadata...")
dataset.records.log(records)

print("=" * 80)
print("Records logged successfully.")
print("Open Argilla and annotate some records:")
print(ARGILLA_API_URL)
print("=" * 80)

Loading or creating Argilla dataset...
Dataset already exists: ag_news
Loading AG News sample from Hugging Face Datasets...
Logging records to Argilla without metadata...


DatasetRecords: The provided batch size 256 was normalized. Using value 50.

Sending records...: 100%|██████████| 1/1 [00:02<00:00,  2.22s/batch]

Records logged successfully.
Open Argilla and annotate some records:
https://parthgeek-argilla-parth-test.hf.space/


In [17]:
print("=" * 80)
print("Filtering completed records...")
print("=" * 80)

status_filter = rg.Query(
    filter=rg.Filter([
        ("status", "==", "completed")
    ])
)

filtered_records = dataset.records(status_filter)

print("Filtered records object created.")
print("If you have not annotated records yet, this may be empty.")
print("=" * 80)

Filtering completed records...
Filtered records object created.
If you have not annotated records yet, this may be empty.


In [18]:
print("=" * 80)
print("Converting completed Argilla records to Hugging Face Dataset...")
print("=" * 80)

hf_dataset = filtered_records.to_datasets()

print("Conversion completed.")
print(hf_dataset)

try:
    print("Number of completed records:", len(hf_dataset))
except Exception as e:
    print("Could not count records directly.")
    print("Error:", repr(e))

print("=" * 80)

Converting completed Argilla records to Hugging Face Dataset...
Conversion completed.
Dataset({
    features: ['id', 'status', '_server_id', 'text', 'label.responses', 'label.responses.users', 'label.responses.status'],
    num_rows: 50
})
Number of completed records: 50


In [19]:
from huggingface_hub import login, whoami

HF_TOKEN = ""  # paste your real HF token here

print("=" * 80)
print("Logging in to Hugging Face...")
print("=" * 80)

login(token=HF_TOKEN)

user_info = whoami(token=HF_TOKEN)

print("Logged in successfully.")
print("Hugging Face username:", user_info["name"])
print("=" * 80)

Logging in to Hugging Face...
Logged in successfully.
Hugging Face username: ParthGeek


In [25]:
from huggingface_hub import login, whoami

HF_TOKEN = ""

print("=" * 80)
print("Logging in to Hugging Face...")
print("=" * 80)

login(token=HF_TOKEN)

user_info = whoami(token=HF_TOKEN)
HF_USERNAME = user_info["name"]

print("Logged in successfully.")
print("Hugging Face username:", HF_USERNAME)
print("=" * 80)


REPO_ID = f"{HF_USERNAME}/ag_news_annotated_completed"

print("=" * 80)
print("Pushing completed records to Hugging Face Hub...")
print("Repo ID:", REPO_ID)
print("=" * 80)

hf_dataset.push_to_hub(
    repo_id=REPO_ID,
    token=HF_TOKEN,
    private=False
)

print("Completed records pushed successfully.")
print("=" * 80)

Logging in to Hugging Face...
Logged in successfully.
Hugging Face username: ParthGeek
Pushing completed records to Hugging Face Hub...
Repo ID: ParthGeek/ag_news_annotated_completed


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 16.5kB / 16.5kB            

Completed records pushed successfully.


In [26]:
FULL_REPO_ID = f"{HF_USERNAME}/ag_news_annotated_full"

print("=" * 80)
print("Pushing full Argilla dataset to Hugging Face Hub...")
print("Repo ID:", FULL_REPO_ID)
print("=" * 80)

dataset.to_hub(
    repo_id=FULL_REPO_ID,
    token=HF_TOKEN
)

print("Full Argilla dataset pushed successfully.")
print("=" * 80)

Pushing full Argilla dataset to Hugging Face Hub...
Repo ID: ParthGeek/ag_news_annotated_full


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 16.4kB / 16.4kB            

Full Argilla dataset pushed successfully.
